# Advance Crime Data Pipeline

## Stage 3: Feature Engineering and Enrichment Layer

**Stakeholder:** Police Force Analytics Unit <br>
**Stage:** 3 of 5: Feature Engineering and Enrichment <br>
**Medallion Layer:** Silver 🥈<br>
**Police Forces:** West Midlands · Thames Valley · Surrey · Dyfed-Powys <br>
**Authors:** Group 1 <br>
**Last Updated:** 21 May 2026

---

### Purpose

This notebook enriches the cleaned Silver crime data by joining three external datasets at the appropriate grain. The output is a fully enriched dataset ready for aggregation in the Gold layer, enabling socioeconomic comparisons across regions.

---

### Enrichment Datasets

| Dataset | Source | Grain | Join Key | Analytical Value |
|---|---|---|---|---|
| Population | ONS Mid-2024 LSOA estimates | LSOA | `lsoa_code` | Enables crime rate normalisation per 1,000 residents |
| House Prices | Land Registry LSOA averages | LSOA × month | `lsoa_code` + `month_num` | Provides socioeconomic context at area level |
| Deprivation (IMD) | MHCLG IMD rankings | Local Authority | `force_name` | Provides deprivation context at force level |

---

### Join Strategy & Assumptions

| Dataset | Join Type | Assumption / Limitation |
|---|---|---|
| Population | LEFT JOIN on `lsoa_code` | Records with suppressed locations (`NOT_RECORDED`) will not match — null population is expected |
| House Prices | LEFT JOIN on `lsoa_code` + `month_num` | Crime data is from 2026 but house prices only available to 2023. **2023 is used as a proxy** — most recent available year. Year is excluded from the join key |
| Deprivation | LEFT JOIN on `force_name` | IMD is England-only — Dyfed-Powys (Wales) will not match and will have null deprivation values. This is a known limitation |

---

### How to Use

**Prerequisites:** `CRIME_PIPELINE.CLEAN.SILVER_CRIME_CLEAN` must be populated. All three enrichment CSVs must be uploaded to `@CRIME_PIPELINE.RAW.ENRICHMENT_STAGE`.

**Hand-off to Gold:** This notebook writes to `CRIME_PIPELINE.CLEAN.SILVER_CRIME_ENRICHED`. The Gold aggregation notebook reads from this table.

## 1. Environment Setup

Initialise the Snowflake session and define all table, stage, and file references in one place.

In [ ]:
import pandas as pd
from snowflake.snowpark.context import get_active_session
from snowflake.connector.pandas_tools import write_pandas

# Establish active Snowpark session
session = get_active_session()
session.sql("USE DATABASE CRIME_PIPELINE").collect()
session.sql("USE SCHEMA CLEAN").collect()

# Table references
SILVER_TABLE   = "CRIME_PIPELINE.CLEAN.SILVER_CRIME_CLEAN"
ENRICHED_TABLE = "SILVER_CRIME_ENRICHED"

# Enrichment stage and file references
STAGE    = "@CRIME_PIPELINE.RAW.ENRICHMENT_STAGE"
POP_FILE = "population_clean.csv"
HP_FILE  = "House_Prices_Silver .csv"
DEP_FILE = "rank results silver stage .csv"

# House price proxy year -- most recent available year in the dataset
# Crime data is from 2026 but house prices only available to 2023
HP_PROXY_YEAR = 2023

print("Session database :", session.get_current_database())
print("Session schema   :", session.get_current_schema())

## 2. Read from Silver Table and Initial Inspection

Load the cleaned crime dataset from `SILVER_CRIME_CLEAN`. A `force_name` column is derived from `falls_within` by stripping the `Police` suffix. This standardises force names to match the deprivation dataset.

In [ ]:
# Read cleaned crime data from Silver table
crime = session.table(SILVER_TABLE).to_pandas()

# Standardise column names
crime.columns = [c.strip().lower().replace(" ", "_") for c in crime.columns]

# Derive force_name from falls_within by stripping ' Police' suffix
# e.g. 'West Midlands Police' -> 'West Midlands'
crime["force_name"] = crime["falls_within"].str.replace(" Police", "", regex=False).str.strip()

# Ensure year and month_num are numeric for join key matching
crime["year"]      = pd.to_numeric(crime["year"],      errors="coerce").astype("Int64")
crime["month_num"] = pd.to_numeric(crime["month_num"], errors="coerce").astype("Int64")

# Record baseline row count for reconciliation report
baseline_count = len(crime)

print("Silver rows read :", len(crime))
print("Force names      :", crime["force_name"].unique().tolist())

In [ ]:
# Visual inspection of the first five rows to confirm is has loaded correctly
crime.head()

## 3. Load Population Data

ONS Mid-2024 LSOA-level population estimates. Used to calculate normalised crime rates (crimes per 1,000 residents) in the Gold layer.

**Grain:** One row per LSOA  
**Join key:** `lsoa_code`

In [ ]:
pop = pd.read_csv(
    session.file.get_stream(f"{STAGE}/{POP_FILE}"),
    dtype={"lsoa_code": str},
    thousands=","
)

# Standardise column names
pop.columns = [c.strip().lower().replace(" ", "_") for c in pop.columns]

# Cast population to integer
pop["population"] = pd.to_numeric(pop["population"], errors="coerce").astype("Int64")

# Validate -- duplicate LSOAs would cause row fan-out on join
dup_lsoa = pop.duplicated(subset=["lsoa_code"]).sum()
print(f"Population rows  : {len(pop):,}")
print(f"Duplicate LSOAs  : {dup_lsoa}")
print(f"Null population  : {pop['population'].isnull().sum()}")
print(pop.head(3))

## 4. Load House Price Data

Land Registry average house prices at LSOA level by month. Provides socioeconomic context at area level.

**Grain:** One row per LSOA × month  
**Join key:** `lsoa_code` + `month_num`  
**Assumption:** House price data is only available up to 2023. The year 2023 is used as a proxy for 2026 prices. Year is excluded from the join key — only `lsoa_code` and `month_num` are used.

In [ ]:
hp = pd.read_csv(
    session.file.get_stream(f"{STAGE}/{HP_FILE}"),
    dtype=str
)

# Standardise column names
hp.columns = [c.strip().lower().replace(" ", "_") for c in hp.columns]

# Cast numeric fields
hp["average_house_price"] = pd.to_numeric(hp["average_house_price"], errors="coerce")
hp["year"]                = pd.to_numeric(hp["year"],                errors="coerce").astype("Int64")

# Filter to proxy year (2023 -- most recent available)
# Then average across all months to get one price per LSOA
# Month is excluded from join key -- crime data month (Jan 2026) has no equivalent in 2023 data
hp_proxy = (
    hp[hp["year"] == HP_PROXY_YEAR]
    .groupby("lsoa_code")["average_house_price"]
    .mean()
    .round(0)
    .reset_index()
    .rename(columns={"average_house_price": "average_house_price_2023"})
)

# Validate -- no duplicates at join grain
dup_hp = hp_proxy.duplicated(subset=["lsoa_code"]).sum()
print(f"House price rows (2023 annual avg): {len(hp_proxy):,}")
print(f"Duplicate LSOAs                   : {dup_hp}")
print(hp_proxy.head(3))

## 5. Load Deprivation Data

Index of Multiple Deprivation (IMD) rankings at local authority level aggregated to police force level. IMD covers England only. Welsh and Northern Irish Police Forces will be not be able to be joined with deprivation data.

**Grain:** One row per local authority → aggregated to force level  
**Join key:** `force_name`  


In [ ]:
dep = pd.read_csv(
    session.file.get_stream(f"{STAGE}/{DEP_FILE}"),
    dtype=str
)

# Standardise column names
dep.columns = [c.strip().lower().replace(" ", "_") for c in dep.columns]

# Cast rank columns to numeric
rank_cols = [c for c in dep.columns if "rank" in c]
dep[rank_cols] = dep[rank_cols].apply(pd.to_numeric, errors="coerce")

# Standardise police_force values to match crime force_name
dep["police_force"] = dep["police_force"].str.strip().str.title()

# Aggregate from LAD level to force level by averaging ranks across LADs
dep_force = dep.groupby("police_force")[rank_cols].mean().round(1).reset_index()
dep_force = dep_force.rename(columns={"police_force": "force_name"})

print(f"Deprivation rows (LAD)   : {len(dep):,}")
print(f"Deprivation rows (force) : {len(dep_force):,}")
print(dep_force)

## 6. Join All Enrichment Datasets

Three sequential LEFT JOINs applied to the crime dataset.

In [ ]:
# ── Join 1: Population on lsoa_code ──────────────────────────────────────────
crime_enriched = crime.merge(
    pop,
    on="lsoa_code",
    how="left"
)

assert len(crime_enriched) == baseline_count, \
    f"Row count mismatch after population join: {len(crime_enriched)} vs {baseline_count}"

print(f"After population join  : {len(crime_enriched):,} rows")
print(f"Null population values : {crime_enriched['population'].isnull().sum():,}")

In [ ]:
# ── Join 2: House Prices on lsoa_code only (2023 annual average proxy) ────────
crime_enriched = crime_enriched.merge(
    hp_proxy,
    on="lsoa_code",
    how="left"
)

assert len(crime_enriched) == baseline_count, \
    f"Row count mismatch after house price join: {len(crime_enriched)} vs {baseline_count}"

print(f"After house price join           : {len(crime_enriched):,} rows")
print(f"Null average_house_price_2023    : {crime_enriched['average_house_price_2023'].isnull().sum():,}")

In [ ]:
# ── Join 3: Deprivation on force_name ────────────────────────────────────────
# Dyfed-Powys will not match -- IMD is England-only
crime_enriched = crime_enriched.merge(
    dep_force,
    on="force_name",
    how="left"
)

assert len(crime_enriched) == baseline_count, \
    f"Row count mismatch after deprivation join: {len(crime_enriched)} vs {baseline_count}"

print(f"After deprivation join : {len(crime_enriched):,} rows")
print(f"Null imd_rank          : {crime_enriched['imd_rank'].isnull().sum():,}")

## 7. Post-Join Reconciliation Report

Document match rates for all three enrichment datasets.

In [ ]:
print("╔══════════════════════════════════════════════════════════════╗")
print("║           ENRICHMENT RECONCILIATION REPORT                  ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"Baseline rows (Silver)       : {baseline_count:,}")
print(f"Rows after all joins         : {len(crime_enriched):,}")

print("\n── Match rates ──")
print(f"Population matched    : {crime_enriched['population'].notnull().sum():,} "
      f"({crime_enriched['population'].notnull().mean()*100:.1f}%)")
print(f"House price matched   : {crime_enriched['average_house_price_2023'].notnull().sum():,} "
      f"({crime_enriched['average_house_price_2023'].notnull().mean()*100:.1f}%)")
print(f"Deprivation matched   : {crime_enriched['imd_rank'].notnull().sum():,} "
      f"({crime_enriched['imd_rank'].notnull().mean()*100:.1f}%)")

print("\n── Unmatched population by force ──")
print(
    crime_enriched[crime_enriched["population"].isnull()]
    ["falls_within"].value_counts()
)

## 8. Inspect Enriched Dataset

In [ ]:
print("Shape  :", crime_enriched.shape)
print("Columns:", crime_enriched.columns.tolist())

crime_enriched.head()

## 9. Export to Enriched Silver Table

Persist the fully enriched dataset to `CRIME_PIPELINE.CLEAN.SILVER_CRIME_ENRICHED` for the final aggregation notebook.

In [ ]:
# Reset index before writing to suppress non-standard index warning
crime_enriched = crime_enriched.reset_index(drop=True)

success, nchunks, nrows, _ = write_pandas(
    conn=session.connection,
    df=crime_enriched,
    table_name=ENRICHED_TABLE,
    database="CRIME_PIPELINE",
    schema="CLEAN",
    auto_create_table=True,
    overwrite=True
)

# Verify written row count matches enriched row count
written_count = session.table(f"CRIME_PIPELINE.CLEAN.{ENRICHED_TABLE}").count()
assert written_count == len(crime_enriched), \
    f"Row count mismatch: {written_count} written vs {len(crime_enriched)} expected"

print(f"Enriched Silver table written successfully")
print(f"Table  : CRIME_PIPELINE.CLEAN.{ENRICHED_TABLE}")
print(f"Rows   : {written_count:,}")

## 10. Export to CSV

Export the enriched dataset as a CSV for local download or further use outside of Snowflake.

In [ ]:
crime_enriched.to_csv("silver_crime_enriched.csv", index=False)

print("Exported : silver_crime_enriched.csv")
print(f"Rows     : {len(crime_enriched):,}")
print(f"Columns  : {len(crime_enriched.columns)}")